In [1]:
import pandas as pd
from tbdynamics.plotting import plot_model_vs_actual
import nevergrad as ng

# Import our convenience wrapper
from estival.wrappers.nevergrad import optimize_model
from tbdynamics.calib_utils import get_bcm, load_targets
from multiprocessing import cpu_count
import plotly.io as pio
pio.templates.default = "plotly_white"

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install m2w64-toolchain`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


In [2]:
params = {
    # "seed_time": 1810.,
    # "start_population_size": 2200000.,
    # "seed_num": 10.,
    # "seed_duration": 3.,
    # "screening_scaleup_shape": 0.1,
    # "screening_inflection_time": 1993.,
    # "screening_end_asymp": 0.6,
}

bcm = get_bcm(params)

In [3]:
num_cores = cpu_count()
# Set the number of workers for parallel optimization
num_workers = num_cores
opt_class = ng.optimizers.TwoPointsDE
orunner = optimize_model(bcm, opt_class=opt_class, num_workers=num_cores)
for i in range(8):
    rec = orunner.minimize(1000)
mle_params = rec.value[1]
mle_params
   

{'start_population_size': 2364553.4734793333,
 'contact_rate': 0.0118013355753768,
 'rr_infection_latent': 0.20203937043583173,
 'rr_infection_recovered': 0.2042564045882957,
 'progression_multiplier': 1.323531334447561,
 'seed_time': 1815.2144014433318,
 'seed_num': 46.77871858286919,
 'seed_duration': 5.140204693021982,
 'smear_positive_death_rate': 0.4091889816705485,
 'smear_negative_death_rate': 0.03467384600133491,
 'smear_positive_self_recovery': 0.2805730380545017,
 'smear_negative_self_recovery': 0.2080226627221957,
 'screening_scaleup_shape': 0.07024543861729088,
 'screening_inflection_time': 1995.1971370747615,
 'screening_end_asymp': 0.5556246947891916}

In [4]:
res = bcm.run(mle_params)
derived_df_0 = res.derived_outputs
targets = load_targets()

In [5]:
plot_model_vs_actual(
    derived_df_0, targets['total_population'], "total_population", "Population", "Modelled vs Data"
)

In [6]:
plot_model_vs_actual(
    derived_df_0, targets['percentage_latent'], "percentage_latent", "Percentage latent", "Modelled vs Data"
)

In [7]:
plot_model_vs_actual(derived_df_0, targets['prevalence_pulmonary'], 'prevalence_pulmonary', 'Infectious prevalence', 'Modelled vs Estimation from 2017 prevalence survey')

In [8]:
plot_model_vs_actual(
    derived_df_0, targets['incidence'], "incidence", "Incidence", "Modelled vs Data"
)

In [9]:
plot_model_vs_actual(
    derived_df_0, targets['notification'], "notification", "Notification", "Modelled vs Data"
)

In [10]:
plot_model_vs_actual(
    derived_df_0, targets['cdr'], "cdr", "Case detection", "Modelled vs Data"
)